# **Projeto Prático: Machine Learning & Inteligência de Mercado**
## **Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)**

---

> **Componente Curricular:** Machine Learning aplicado à Administração  
> **Instituição:** Curso de Graduação em Administração  
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos (patrimônio, audiovisual, design, música, software, livros e arquitetura). Cada aula entrega uma peça do pipeline (limpeza → estatística → modelagem → rede → texto → storytelling), que alimenta, ao final, um painel executivo de (re)concentração de mercado.

* **Diretriz de Execução:** Execute as células sequencialmente e utilize os enunciados propostos ao final de cada bloco para interagir com a IA na resolução dos desafios analíticos e na interpretação dos resultados sob a ótica de negócios.
* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** [Paper OpenFCS (HAL)](https://hal.science/hal-05712802v1) e apostila *Economia Criativa em Dados*.

## **Aula 3 — Bases de Dados e SQL: Modelando o Acervo como Estrela**
Nesta aula aprendemos a pensar em dados como um banco relacional — tabela fato e tabelas dimensão — e a consultá-los com SQL, a língua universal de qualquer banco de dados corporativo.

### **Recarregar o acervo + instalar DuckDB**

In [2]:
import requests, zipfile, io, os
import pandas as pd

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = 'openfcs/openfcs-1.0.0/data/derived/'
edges = pd.read_csv(endereco + "trade_edges.csv")
entities = pd.read_csv(endereco + "entities.csv")

!pip install duckdb --quiet
import duckdb

print(f"trade_edges.csv: {edges.shape[0]:,} linhas")

trade_edges.csv: 2,197,978 linhas


### **3.1 Por que SQL, se já temos pandas?**
Pandas resolve tudo dentro de um notebook. Mas a maior parte dos dados de uma empresa não vive em um CSV — vive em um banco de dados, acessado por SQL. Analista de dados que só sabe pandas fica dependente de alguém que "exporte a planilha" para ele. Quem sabe SQL consulta a fonte diretamente.

**Repetir o filtro de resolução da Aula 2 — base para a tabela fato**

In [3]:
edges.head()

,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
0,1,South America,Other territories,Manufacturing of crafts and design goods,2002,2.026,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
1,2,South America,Other territories,Carpets,2002,0.021,Exports,craft_sub,CER020s,C. Visual arts (crafts) / F. Design,provisional
2,3,South America,Other territories,Fashion accessories,2002,0.116,Exports,craft_sub,CER020s,C. Visual arts (crafts) / F. Design,provisional
3,4,South America,Other territories,Interior,2002,0.749,Exports,craft_sub,CER020s,C. Visual arts (crafts) / F. Design,provisional
4,5,South America,Other territories,Jewellery,2002,0.100,Exports,craft_sub,CER020s,C. Visual arts (crafts) / F. Design,provisional


In [4]:
print(edges["resolution"].value_counts())

# Valor confirmado: "cer7" é a resolução dos sete domínios
edges7 = edges[edges["resolution"] == "cer7"].copy()
print(f"Tabela fato (edges7): {len(edges7):,} linhas")   # deve dar 1.026.400

resolution
craft_sub    1171578
cer7         1026400
Name: count, dtype: int64
Tabela fato (edges7): 1,026,400 linhas


### **3.2 Modelando o acervo como estrela: fato e dimensões**
Em um **modelo em estrela**, existe uma tabela **fato** no centro (o que se mede, o que se soma) cercada por tabelas **dimensão** (o contexto pelo qual se agrupa e filtra). Vamos construir as três dimensões principais do nosso acervo.

#### **construir as dimensões**

In [5]:
# Dimensão Economias: extraída diretamente da tabela fato já limpa
dim_economias = (
    edges7[["economy"]]
    .drop_duplicates()
    .sort_values("economy")
    .reset_index(drop=True)
)

# Dimensão Domínios: os sete domínios do FCS presentes na base
dim_dominios = (
    edges7[["fcs_domain"]]
    .drop_duplicates()
    .sort_values("fcs_domain")
    .reset_index(drop=True)
)

# Dimensão Anos: intervalo coberto
dim_anos = pd.DataFrame({"year": sorted(edges7["year"].unique())})

print(f"dim_economias: {len(dim_economias)} linhas")
print(f"dim_dominios:  {len(dim_dominios)} linhas")
print(f"dim_anos:      {len(dim_anos)} linhas")

dim_economias: 204 linhas
dim_dominios:  6 linhas
dim_anos:      23 linhas


In [6]:
print(dim_dominios['fcs_domain'])

0                   A. Cultural and natural heritage
1    B. Performance and celebration / C. Visual arts
2                C. Visual arts (crafts) / F. Design
3                                 D. Books and press
4               E. Audiovisual and interactive media
5                    F. Design and creative services
Name: fcs_domain, dtype: object


### **3.3 Primeiras consultas: SELECT, WHERE, ORDER BY**
O DuckDB consegue enxergar DataFrames do pandas diretamente pelo nome da variável — não é preciso importar os dados para dentro de um "banco" separado.

In [7]:
# SELECT + WHERE + ORDER BY, direto sobre o DataFrame edges7
consulta = """
SELECT *
FROM edges7
"""
duckdb.sql(consulta).df()

,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
0,1,South America,Other territories,Manufacturing of crafts and design goods,2002,2.026,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
1,8,South America,Other territories,Books and publishing,2002,3.151,Exports,cer7,CER030,D. Books and press,confirmed
2,9,South America,Other territories,"Music, performing and visual arts",2002,0.006,Exports,cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle
3,10,South America,Other territories,"Software, video games and recorded media",2002,0.002,Exports,cer7,CER060,E. Audiovisual and interactive media,provisional
4,11,South America,World n.e.s.,"Audiovisual, multimedia and photography",2002,0.000,Exports,cer7,CER010,E. Audiovisual and interactive media,confirmed
...,...,...,...,...,...,...,...,...,...,...,...
1026395,2197967,G-77 (Group of 77),Zimbabwe,Manufacturing of crafts and design goods,2024,42.898,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
1026396,2197975,G-77 (Group of 77),Zimbabwe,Books and publishing,2024,4.011,Exports,cer7,CER030,D. Books and press,confirmed
1026397,2197976,G-77 (Group of 77),Zimbabwe,"Music, performing and visual arts",2024,0.977,Exports,cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle
1026398,2197977,G-77 (Group of 77),Zimbabwe,"Software, video games and recorded media",2024,5.491,Exports,cer7,CER060,E. Audiovisual and interactive media,provisional


In [9]:
# SELECT + WHERE + ORDER BY, direto sobre o DataFrame edges7
consulta = """
SELECT distinct(fcs_domain)
FROM edges7
"""
duckdb.sql(consulta).df()

,fcs_domain
0,B. Performance and celebration / C. Visual arts
1,C. Visual arts (crafts) / F. Design
2,F. Design and creative services
3,A. Cultural and natural heritage
4,D. Books and press
5,E. Audiovisual and interactive media


In [10]:
# SELECT + WHERE + ORDER BY, direto sobre o DataFrame edges7
consulta = """
SELECT economy, partner, year, value_usd_millions
FROM edges7
WHERE fcs_domain = 'E. Audiovisual and interactive media'
  AND year = 2024
ORDER BY value_usd_millions DESC
LIMIT 10
"""
duckdb.sql(consulta).df()

,economy,partner,year,value_usd_millions
0,G-77 (Group of 77),United States,2024,9903.889
1,Republic of Korea,United States,2024,5445.818
2,United States,Mexico,2024,5345.757
3,"China, Hong Kong SAR",G-77 (Group of 77),2024,5234.061
4,China,United States,2024,4398.690
5,"China, Hong Kong SAR",China,2024,4102.316
6,"China, Taiwan Province of",G-77 (Group of 77),2024,4009.736
7,G-77 (Group of 77),China,2024,3861.939
8,Malaysia,G-77 (Group of 77),2024,3598.826
9,"China, Taiwan Province of",China,2024,3486.821


### **3.4 Agregações em SQL: GROUP BY**
Compare esta consulta com o `groupby` que fizemos na Aula 2 — o resultado deve ser idêntico. É uma boa forma de validar se você entendeu as duas ferramentas.

In [11]:
consulta_agregada = """
SELECT fcs_domain, year, SUM(value_usd_millions) AS total_exportado
FROM edges7
GROUP BY fcs_domain, year
ORDER BY total_exportado DESC
LIMIT 10
"""
resultado_sql = duckdb.sql(consulta_agregada).df()
resultado_sql

,fcs_domain,year,total_exportado
0,C. Visual arts (crafts) / F. Design,2022,1014688.005
1,C. Visual arts (crafts) / F. Design,2023,993887.877
2,C. Visual arts (crafts) / F. Design,2024,950502.764
3,C. Visual arts (crafts) / F. Design,2021,931621.121
4,C. Visual arts (crafts) / F. Design,2014,848209.854
5,C. Visual arts (crafts) / F. Design,2019,843095.487
6,C. Visual arts (crafts) / F. Design,2018,800094.833
7,C. Visual arts (crafts) / F. Design,2017,766687.087
8,C. Visual arts (crafts) / F. Design,2013,764976.753
9,C. Visual arts (crafts) / F. Design,2015,762359.756


#### **validação cruzada com o pandas da Aula 2**

In [12]:
resultado_pandas = (
    edges7.groupby(["fcs_domain", "year"])["value_usd_millions"]
    .sum()
    .reset_index()
    .sort_values("value_usd_millions", ascending=False)
    .head(10)
)

# Os dois devem bater
print(resultado_sql["total_exportado"].round(2).tolist())
print(resultado_pandas["value_usd_millions"].round(2).tolist())

[1014688.0, 993887.88, 950502.76, 931621.12, 848209.85, 843095.49, 800094.83, 766687.09, 764976.75, 762359.76]
[1014688.0, 993887.88, 950502.76, 931621.12, 848209.85, 843095.49, 800094.83, 766687.09, 764976.75, 762359.76]


### **3.5 JOIN: conectando o fato às dimensões**
Uma tabela fato normalmente guarda apenas **chaves** (nomes, códigos). Atributos descritivos adicionais vêm de tabelas dimensão através de um `JOIN`. Vamos usar `products_crosswalk.csv` para isso — mas antes, inspecione as colunas: **nunca escreva um JOIN sem antes conferir os nomes exatos das colunas.**

#### **inspecionar antes de unir — regra de ouro**

In [13]:
crosswalk = pd.read_csv(endereco + "products_crosswalk.csv")
print(crosswalk.columns.tolist())
crosswalk.head()

['product', 'resolution', 'cer', 'fcs', 'status', 'parent']


,product,resolution,cer,fcs,status,parent
0,"Audiovisual, multimedia and photography",cer7,CER010,E. Audiovisual and interactive media,confirmed,NaN
1,Manufacturing of crafts and design goods,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle,NaN
2,Books and publishing,cer7,CER030,D. Books and press,confirmed,NaN
3,"Music, performing and visual arts",cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle,NaN
4,Architecture,cer7,CER050,F. Design and creative services,confirmed,NaN


#### **JOIN com base no que foi observado acima**

In [15]:
# Ajuste os nomes de coluna conforme o que apareceu na célula anterior
consulta_join = """
SELECT e.economy, e.partner, e.year, e.fcs_domain, e.value_usd_millions,
       c.status
FROM edges7 e
LEFT JOIN crosswalk c
  ON e.cer_code = c.cer
LIMIT 10
"""
duckdb.sql(consulta_join).df()

,economy,partner,year,fcs_domain,value_usd_millions,status
0,South America,Other territories,2002,C. Visual arts (crafts) / F. Design,2.026,provisional_straddle
1,South America,Other territories,2002,D. Books and press,3.151,confirmed
2,South America,Other territories,2002,B. Performance and celebration / C. Visual arts,0.006,provisional_straddle
3,South America,Other territories,2002,E. Audiovisual and interactive media,0.002,provisional
4,South America,World n.e.s.,2002,E. Audiovisual and interactive media,0.000,confirmed
5,South America,World n.e.s.,2002,C. Visual arts (crafts) / F. Design,0.000,provisional_straddle
6,South America,World n.e.s.,2002,D. Books and press,0.000,confirmed
7,South America,World n.e.s.,2002,B. Performance and celebration / C. Visual arts,0.000,provisional_straddle
8,South America,World n.e.s.,2002,F. Design and creative services,0.000,confirmed
9,South America,World n.e.s.,2002,E. Audiovisual and interactive media,0.000,provisional


### **3.6 Um comando de cada família: DDL, DML e DQL**
- **DDL** (Data Definition Language): `CREATE`, `DROP` — define estruturas.
- **DML** (Data Manipulation Language): `INSERT`, `UPDATE`, `DELETE` — altera dados.
- **DQL** (Data Query Language): `SELECT` — consulta dados (99% do trabalho de um analista).

Vamos ver DDL/DML funcionando em uma tabela minúscula e descartável — nunca em `edges7`.

#### **demonstração isolada de DDL/DML**

In [16]:
duckdb.sql("CREATE TABLE demo (pais VARCHAR, valor DOUBLE)")
duckdb.sql("INSERT INTO demo VALUES ('Brazil', 120.5), ('Japan', 340.2)")
print(duckdb.sql("SELECT * FROM demo").df())

duckdb.sql("UPDATE demo SET valor = 999 WHERE pais = 'Brazil'")
duckdb.sql("DELETE FROM demo WHERE pais = 'Japan'")
print(duckdb.sql("SELECT * FROM demo").df())

duckdb.sql("DROP TABLE demo")

     pais  valor
0  Brazil  120.5
1   Japan  340.2
     pais  valor
0  Brazil  999.0


# **Exercícios**

## 🧪 Exercícios Práticos — Aula 3

> **Como usar:** resolva cada exercício em uma célula de código abaixo do enunciado. Depois, leve o resultado para uma IA usando o *prompt sugerido*. Cole a resposta da IA em uma célula de texto e escreva, em 2-3 linhas, se você concorda com ela e por quê.

---

### Exercício 1 — SELECT + WHERE
**🎯 Objetivo:** praticar filtragem em SQL.

**📝 Tarefa:** Escreva uma consulta SQL que retorne todos os fluxos de exportação do domínio "Books/Press" (ou equivalente) no ano de 2020, ordenados do maior para o menor valor.

**🤖 Pergunte à IA:**
> "Escrevi esta consulta SQL: [cole sua query]. Ela está correta para responder 'quais foram os maiores exportadores de livros e imprensa em 2020'? Sugira uma melhoria se houver."

---

### Exercício 2 — GROUP BY + ranking
**🎯 Objetivo:** praticar agregação e ordenação em SQL.

**📝 Tarefa:** Escreva uma consulta que retorne os 10 maiores exportadores (`economy`) em valor total acumulado, somando todos os domínios e anos.

**🤖 Pergunte à IA:**
> "Estes são os 10 maiores exportadores acumulados de bens criativos: [cole a lista]. Baseado no seu conhecimento geral de economia, algum desses países te surpreende estar (ou não estar) nessa lista? Por quê?"

---

### Exercício 3 — JOIN
**🎯 Objetivo:** praticar junção entre tabela fato e dimensão.

**📝 Tarefa:** Usando o `JOIN` da Célula 14 como modelo, escreva uma consulta que traga, para cada linha da tabela fato, alguma informação adicional do `products_crosswalk.csv` que ainda não esteja em `edges7`.

**🤖 Pergunte à IA:**
> "Fiz um JOIN entre uma tabela fato de comércio e uma tabela de dimensão de produtos: [cole a query]. Do ponto de vista de modelagem de dados, esse é um LEFT JOIN, INNER JOIN ou outro tipo? Qual a diferença prática entre eles nesse caso?"

---

### Exercício 4 — SQL vs. pandas: o mesmo resultado, dois caminhos
**🎯 Objetivo:** validar consistência entre ferramentas.

**📝 Tarefa:** Escolha uma pergunta de negócio (ex.: "total exportado pelo Brasil em 2023") e responda-a duas vezes: uma com `pandas`, outra com SQL via DuckDB. Os resultados batem?

**🤖 Pergunte à IA:**
> "Resolvi a mesma pergunta de negócio com pandas e com SQL e obtive [resultado 1] e [resultado 2]. Eles batem? Se não baterem, que tipo de erro costuma causar essa divergência?"

---

### Exercício 5 — Desenhe seu próprio esquema estrela
**🎯 Objetivo:** transferir o conceito de modelagem para um problema novo.

**📝 Tarefa:** Sem escrever código, descreva em texto um esquema em estrela para responder a esta pergunta: *"Quais países parceiros (importadores) mais dependem de um único fornecedor em cada domínio?"* Qual seria a tabela fato? Quais dimensões você usaria?

**🤖 Pergunte à IA:**
> "Quero montar um esquema em estrela para responder: [cole a pergunta]. Minha proposta de fato e dimensões é: [cole sua proposta]. Isso faz sentido do ponto de vista de modelagem, ou você sugeriria outra estrutura?"

---

> 💡 **Dica geral:** o Exercício 5 é a base conceitual da Aula 15, onde vamos montar o painel executivo final.